# TUDataset OHSU Graph Classification with GNNVisualizer

This notebook trains graph-level **GCN**, **GraphSAGE**, **GAT**, and **GIN** models on the PyTorch Geometric `TUDataset(name="OHSU")` benchmark, then renders each trained model with `GNNVisualizer` on the same input graph.

OHSU is a small bioinformatics/neuroscience graph classification dataset. The TU Dortmund dataset table reports an average of about 82 nodes and 200 edges per graph, so it is a good fit for testing mid-sized graph visualization around the 100-node scale.

Source docs: [PyG TUDataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.TUDataset.html) and [TU Dortmund graph datasets](https://chrsmrrs.github.io/datasets/docs/datasets/).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Optional environment variables: `OHSU_EPOCHS`, `OHSU_TARGET_NODES`, and `OHSU_HIDDEN_CHANNELS`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer, GraphEditor, GraphVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("OHSU_EPOCHS", "80"))
TARGET_NODES = int(os.environ.get("OHSU_TARGET_NODES", "100"))
HIDDEN_CHANNELS = int(os.environ.get("OHSU_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 16


def prepare_graph(data):
    data = data.clone()
    if getattr(data, "x", None) is None:
        degree = torch.bincount(data.edge_index[0], minlength=data.num_nodes).float().view(-1, 1)
        data.x = degree / degree.max().clamp_min(1.0)
    else:
        data.x = data.x.float()
    data.y = data.y.view(-1).long()
    return data


dataset = TUDataset(root=str(repo_root / "data" / "tudataset"), name="OHSU")
graphs = [prepare_graph(dataset[index]) for index in range(len(dataset))]
generator = torch.Generator().manual_seed(SEED)
order = torch.randperm(len(graphs), generator=generator).tolist()
train_size = max(1, int(0.8 * len(order)))
train_graphs = [graphs[index] for index in order[:train_size]]
valid_graphs = [graphs[index] for index in order[train_size:]] or train_graphs[:1]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)

visual_index = min(range(len(graphs)), key=lambda index: abs(graphs[index].num_nodes - TARGET_NODES))
visual_data = graphs[visual_index]
query_pair = [0, min(visual_data.num_nodes - 1, max(1, visual_data.num_nodes // 2))]
num_features = visual_data.num_features
num_classes = int(dataset.num_classes)

node_counts = torch.tensor([graph.num_nodes for graph in graphs], dtype=torch.float)
edge_counts = torch.tensor([graph.edge_index.size(1) for graph in graphs], dtype=torch.float)
display(Markdown(
    f"OHSU loaded with **{len(dataset)} graphs**. "
    f"This notebook trains on **{len(train_graphs)} graphs** and validates on **{len(valid_graphs)} graphs**. "
    f"Mean graph size in the local copy is **{node_counts.mean():.1f} nodes** and **{edge_counts.mean():.1f} directed edges**. "
    f"The visualized graph is index `{visual_index}` with **{visual_data.num_nodes} nodes**, "
    f"**{visual_data.edge_index.size(1)} directed edges**, and **{num_features} node features**."
))

## Input Graph Views

The next cells render the exact OHSU graph used by the model visualization. `GraphVisualizer` gives a read-only graph/matrix overview, while `GraphEditor` shows the same input graph in the editable graph surface.

In [ ]:
def graph_json_from_pyg_data(data):
    return {
        "x": data.x.detach().cpu().tolist(),
        "edge_index": data.edge_index.detach().cpu().tolist(),
        "edge_attr": getattr(data, "edge_attr", None).detach().cpu().tolist()
        if getattr(data, "edge_attr", None) is not None else [],
        "y": data.y.detach().cpu().view(-1).tolist() if getattr(data, "y", None) is not None else [],
        "batch": [0] * int(data.num_nodes),
    }


input_graph_json = graph_json_from_pyg_data(visual_data)

graph_visualizer = GraphVisualizer()
graph_visualizer.graphData = input_graph_json

graph_editor = GraphEditor()
graph_editor.graphData = input_graph_json

assert len(graph_visualizer.graphData["x"]) == visual_data.num_nodes
assert len(graph_editor.graphData["x"]) == visual_data.num_nodes
assert graph_visualizer.graphData["edge_index"] == graph_editor.graphData["edge_index"]
assert graph_visualizer.renderer == "auto"

display(Markdown(
    "| Input graph view | Value |\n"
    "|---|---:|\n"
    f"| Source graph index | {visual_index} |\n"
    f"| Nodes | {len(input_graph_json['x'])} |\n"
    f"| Directed edges | {len(input_graph_json['edge_index'][0])} |\n"
    f"| Node feature width | {len(input_graph_json['x'][0]) if input_graph_json['x'] else 0} |\n"
    "| Shared graph source | `visual_data` |"
))

In [ ]:
display(graph_visualizer)

In [ ]:
display(graph_editor)

In [ ]:
class BaseOHSUClassifier(nn.Module):
    def graph_batch(self, x, batch):
        if batch is None:
            return torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return batch

    def pool_and_classify(self, h, batch):
        return self.classifier(global_mean_pool(h, batch))


class OHSUGCN(BaseOHSUClassifier):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        batch = self.graph_batch(x, batch)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.pool_and_classify(h, batch)


class OHSUGraphSAGE(BaseOHSUClassifier):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        batch = self.graph_batch(x, batch)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.pool_and_classify(h, batch)


class OHSUGAT(BaseOHSUClassifier):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        batch = self.graph_batch(x, batch)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.pool_and_classify(h, batch)


class OHSUGIN(BaseOHSUClassifier):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        batch = self.graph_batch(x, batch)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.pool_and_classify(h, batch)


MODEL_BUILDERS = {
    "GCN": OHSUGCN,
    "GraphSAGE": OHSUGraphSAGE,
    "GAT": OHSUGAT,
    "GIN": OHSUGIN,
}

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            pred = logits.argmax(dim=1)
            target = batch.y.view(-1).long()
            correct += int((pred == target).sum())
            total += int(target.numel())
    return correct / max(total, 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(logits, batch.y.view(-1).long())
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {
        "final_loss": losses[-1] if losses else float("nan"),
        "train_accuracy": accuracy(model, train_loader),
        "valid_accuracy": accuracy(model, valid_loader),
    }


def fit_model(name):
    torch.manual_seed(SEED)
    candidate = MODEL_BUILDERS[name](num_features, HIDDEN_CHANNELS, num_classes)
    candidate_metrics = train_model(candidate, train_loader)
    candidate.eval()
    return candidate, candidate_metrics


models = {}
metrics_by_model = {}
for model_name in MODEL_BUILDERS:
    models[model_name], metrics_by_model[model_name] = fit_model(model_name)

model = models["GAT"]
metrics = metrics_by_model["GAT"]

rows = [
    "| Model | Final training loss | Train accuracy | Validation accuracy |",
    "|---|---:|---:|---:|",
]
for model_name, model_metrics in metrics_by_model.items():
    rows.append(
        f"| {model_name} | {model_metrics['final_loss']:.4f} | "
        f"{model_metrics['train_accuracy']:.3f} | {model_metrics['valid_accuracy']:.3f} |"
    )
display(Markdown("\n".join(rows)))

The next cell builds one `GNNVisualizer` widget for each trained model. Every model is visualized on the same OHSU `visual_data` graph shown above by `GraphVisualizer` and `GraphEditor`.

In [ ]:
EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GraphSAGE": "SAGEConv",
    "GAT": "GATConv",
    "GIN": "GINConv",
}
EXPECTED_AGGREGATIONS = {
    "GCN": "gcn-normalized",
    "GraphSAGE": "mean",
    "GAT": "attention",
    "GIN": "sum",
}

visualizers = {}
summary_rows = [
    "| Model | First layer | Aggregation | Graph pooling | Hidden width | Visualized nodes | Query |",
    "|---|---|---|---|---:|---:|---|",
]
for model_name, candidate in models.items():
    widget = GNNVisualizer()
    widget.add_model(
        data=visual_data,
        model=candidate.eval(),
        subgraphSample=False,
        queries=[query_pair],
        mode="graph",
    )
    assert widget.modelInfo["conv1"]["type"] == EXPECTED_LAYER_TYPES[model_name]
    assert widget.modelInfo["conv1"].get("aggregation") == EXPECTED_AGGREGATIONS[model_name]
    assert widget.graphData["edge_index"] == input_graph_json["edge_index"]
    assert len(widget.graphData["x"]) == visual_data.num_nodes
    assert "graphAggregation" in widget.intmData
    assert len(widget.intmData["act1"][0]) == HIDDEN_CHANNELS
    visualizers[model_name] = widget
    summary_rows.append(
        f"| {model_name} | `{widget.modelInfo['conv1']['type']}` | "
        f"`{widget.modelInfo['conv1'].get('aggregation')}` | "
        f"`{widget.intmData['graphAggregation']['type']}` | "
        f"{len(widget.intmData['act1'][0])} | {len(widget.graphData['x'])} | "
        f"`{widget.queries}` |"
    )

display(Markdown("\n".join(summary_rows)))

## GCN

In [ ]:
display(visualizers["GCN"])

## GraphSAGE

In [ ]:
display(visualizers["GraphSAGE"])

## GAT

In [ ]:
display(visualizers["GAT"])

## GIN

In [ ]:
display(visualizers["GIN"])